# Notebook 12 — Train, Validation & Test Data
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])
df_encoded = df.copy()
for col in encode_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

X = df_encoded.drop(columns=['customerID', 'Churn'])
y = (df['Churn'] == 'Yes').astype(int)
print(f"Full feature set ready: {X.shape}")


Full feature set ready: (7043, 19)


/tmp/ipykernel_813/2990341750.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])


---
## 1-3. Training, Validation, and Test Datasets

### Understand
(Recap from Sprint 5, Notebook 1) **Training data** is what the model directly learns
from. **Validation data** is used to tune choices (hyperparameters, feature sets) without
touching the test set. **Test data** is used exactly once, at the very end, for an honest
final performance estimate.

### Demonstrate
**Why three, not two?** If I only had train/test and tuned hyperparameters by repeatedly
checking test performance, the test set would stop being a genuinely unseen estimate —
I'd be *indirectly* fitting to it through my tuning choices. Validation data exists
specifically to absorb that repeated-checking role instead.


---
## 4. Train-Test Split & 5. Train-Validation-Test Split

### Implement


In [2]:
# Two-step split: first carve out the test set, then split the remainder into train/val
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

for name, subset in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f"{name:<12}: {len(subset):,} rows ({len(subset)/len(X)*100:.1f}%), churn rate={subset.mean()*100:.2f}%")


Train       : 4,929 rows (70.0%), churn rate=26.54%
Validation  : 1,057 rows (15.0%), churn rate=26.58%
Test        : 1,057 rows (15.0%), churn rate=26.49%


---
## 6. Random State

### Understand
`random_state` fixes the pseudo-random split so the exact same rows end up in
train/val/test every time the code runs — essential for reproducibility (Sprint 3,
Notebook 3's "Random Seed" topic, applied here specifically to splitting).

### Implement


In [3]:
split_a = train_test_split(X, y, test_size=0.2, random_state=42)[1].index
split_b = train_test_split(X, y, test_size=0.2, random_state=42)[1].index
split_c = train_test_split(X, y, test_size=0.2, random_state=99)[1].index

print(f"Same random_state (42) twice -> identical split: {(split_a == split_b).all()}")
print(f"Different random_state (99) -> different split : {(split_a == split_c).all()}")


Same random_state (42) twice -> identical split: True
Different random_state (99) -> different split : False


---
## 7. Stratified Sampling

### Understand
(Recap from Sprint 5, Notebook 1) `stratify=y` preserves the target's class ratio in
every split — critical here given the documented 2.77:1 imbalance.

### Demonstrate — the Risk of NOT Stratifying


In [4]:
np.random.seed(0)
unstratified_rates = []
stratified_rates = []
for seed in range(10):
    _, test_unstrat, _, y_test_unstrat = train_test_split(X, y, test_size=0.1, random_state=seed)
    _, test_strat, _, y_test_strat = train_test_split(X, y, test_size=0.1, stratify=y, random_state=seed)
    unstratified_rates.append(y_test_unstrat.mean())
    stratified_rates.append(y_test_strat.mean())

print(f"True churn rate: {y.mean()*100:.2f}%")
print(f"Unstratified test-set churn rate across 10 random splits: {np.min(unstratified_rates)*100:.2f}% to {np.max(unstratified_rates)*100:.2f}% (range: {(np.max(unstratified_rates)-np.min(unstratified_rates))*100:.2f} points)")
print(f"Stratified test-set churn rate across 10 random splits  : {np.min(stratified_rates)*100:.2f}% to {np.max(stratified_rates)*100:.2f}% (range: {(np.max(stratified_rates)-np.min(stratified_rates))*100:.2f} points)")


True churn rate: 26.54%
Unstratified test-set churn rate across 10 random splits: 23.26% to 29.36% (range: 6.10 points)
Stratified test-set churn rate across 10 random splits  : 26.52% to 26.52% (range: 0.00 points)


**Finding:** Unstratified splits show real, measurable variation in test-set churn
rate purely from random luck; stratified splits stay essentially fixed at the true
26.54% rate. With an imbalanced minority class, this variation isn't cosmetic — it
directly changes how many minority examples land in the test set.


---
## 8. Data Leakage (Preview — Full Treatment in Notebook 13)

### Understand
The single most common data-leakage mistake happens right here, at the splitting stage:
fitting a scaler or encoder on the FULL dataset before splitting lets information from
the test set influence the training process.

### Demonstrate — Incorrect vs Correct Workflow


In [5]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# INCORRECT: fit the scaler on the FULL dataset, THEN split
scaler_wrong = StandardScaler().fit(df[numeric_cols])   # sees train AND test data
X_scaled_wrong = scaler_wrong.transform(df[numeric_cols])
print(f"INCORRECT workflow -> scaler's learned mean (uses ALL 7,043 rows): {scaler_wrong.mean_.round(2)}")

# CORRECT: split FIRST, fit the scaler ONLY on the training portion
train_idx, test_idx = train_test_split(df.index, test_size=0.2, stratify=y, random_state=42)
scaler_right = StandardScaler().fit(df.loc[train_idx, numeric_cols])   # sees ONLY training data
print(f"CORRECT workflow   -> scaler's learned mean (uses ONLY {len(train_idx):,} training rows): {scaler_right.mean_.round(2)}")

print(f"\nDifference exists: {not np.allclose(scaler_wrong.mean_, scaler_right.mean_)}")


INCORRECT workflow -> scaler's learned mean (uses ALL 7,043 rows): [  32.37   64.76 2279.73]
CORRECT workflow   -> scaler's learned mean (uses ONLY 5,634 training rows): [  32.49   64.93 2299.33]

Difference exists: True


**Finding:** The two scalers learn measurably different means — the "incorrect"
version has already absorbed information from rows that should have been held out as
unseen test data. This is a real, numerically demonstrated example of train-test
contamination, covered in full in Notebook 13.


---
## 9. Temporal Splitting

### Understand
For time-ordered data, a random split is wrong — it can let "future" rows train a model
that's evaluated on "past" rows, which is impossible in real deployment (a model can
never see the future when trained). The right approach is a temporal split: train on
earlier data, test on later data.

### Demonstrate


In [6]:
print("This dataset has NO date/time column (confirmed Sprint 4, Notebook 2) —")
print("every row is an independent customer snapshot, not part of a time sequence.")
print("Random stratified splitting (used throughout this notebook) is therefore appropriate;")
print("temporal splitting does not apply here, but would be MANDATORY for time-series data")
print("(e.g., predicting next month's churn using this month's data as training).")


This dataset has NO date/time column (confirmed Sprint 4, Notebook 2) —
every row is an independent customer snapshot, not part of a time sequence.
Random stratified splitting (used throughout this notebook) is therefore appropriate;
temporal splitting does not apply here, but would be MANDATORY for time-series data
(e.g., predicting next month's churn using this month's data as training).


---
## 10. Why Preprocessing Must Be Fitted Only on Training Data

### Understand
Every preprocessing step that "learns" something from data — a scaler's mean/std, an
encoder's category list, an imputer's fill value — must be fit ONLY on the training set,
then applied (never re-fit) to validation/test. Fitting on the full dataset leaks test
information into training, producing an overoptimistic performance estimate that won't
hold up in real deployment.

### Demonstrate — Correct Full Workflow


In [7]:
# CORRECT workflow, start to finish
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numeric_cols])   # FIT on train
X_test_scaled = scaler.transform(X_test[numeric_cols])          # TRANSFORM ONLY on test — never fit again

model = LogisticRegression(max_iter=2000).fit(X_train_scaled, y_train)
print(f"Test accuracy (scaler correctly fit on TRAIN ONLY): {model.score(X_test_scaled, y_test):.4f}")


Test accuracy (scaler correctly fit on TRAIN ONLY): 0.7722


### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** Any preprocessing step fit before splitting risks leaking test
  information into training.
- **Analysis:** Demonstrated numerically (Topic 8) that a scaler fit on the full
  dataset learns a measurably different mean than one fit correctly on the training
  fold alone.
- **Technique Selected:** Strict "split first, fit second" ordering for every future
  preprocessing step in this sprint.
- **Reason:** The only way to guarantee the test set genuinely represents unseen data.
- **Implementation:** shown throughout this notebook.
- **Result:** A reproducible, stratified 70/15/15 train/validation/test split with all
  preprocessing correctly scoped to the training fold.
- **Impact:** Every performance number reported from this point forward in the sprint
  reflects a realistic, non-leaked estimate.


---
## Summary

| Concept | This Dataset's Application |
|---|---|
| Train/Val/Test | Stratified 70/15/15 split |
| Random State | Fixed (42) for full reproducibility |
| Stratified Sampling | Essential — demonstrated real variance reduction across 10 trials |
| Temporal Splitting | Not applicable — no date/time column exists |
| Fit-on-train discipline | Demonstrated numerically: leaky vs. correct scaler produce different means |

**Next notebook:** `13_Data_Leakage.ipynb` — a full, dedicated treatment of every leakage
type, building directly on Topic 8-10's preview here.
